[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 Medium: Adam Optimizer

Реализуйте **Adam** с нуля. Adam объединяет momentum по градиенту и адаптивный масштаб шага по квадратам градиента. Для каждого параметра optimizer хранит собственное состояние, поэтому это уже не одна формула обновления, а небольшой stateful-объект.

### Интуиция двух моментов
- `m` — экспоненциальное среднее градиентов: сглаживает шум и задаёт устойчивое направление;
- `v` — экспоненциальное среднее квадратов градиентов: оценивает типичный масштаб каждой координаты;
- деление на `sqrt(v)` уменьшает шаг для координат с крупными градиентами и увеличивает относительный шаг для малых;
- `eps` защищает знаменатель от деления на ноль.

### Зачем нужна bias correction
`m` и `v` инициализируются нулями, поэтому первые экспоненциальные средние смещены к нулю. Счётчик шага `t` и множители `1-β₁ᵗ`, `1-β₂ᵗ` исправляют это смещение. Без correction первые обновления не совпадут с Adam из PyTorch.

### Состояние optimizer
Для каждого параметра нужны тензоры `m` и `v` той же формы, dtype и device. `zero_grad()` очищает только `.grad`; моменты и номер шага между итерациями сохраняются. Параметры без gradient следует пропускать.

### Контракт
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### Алгоритм для каждого параметра
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

Обновляйте параметры без построения нового autograd-графа. Не используйте `.data` как универсальный способ обучения в прикладном коде; здесь важно понять границу между вычислением gradients и шагом optimizer.

### Быстрые самопроверки
- после одного шага параметры изменяются;
- пять последовательных шагов совпадают с `torch.optim.Adam` при одинаковых настройках;
- `zero_grad()` обнуляет gradients, но не моменты;
- `m` и `v` имеют форму соответствующего параметра.

### Типичные ошибки
- один общий `m` или `v` для параметров разных форм;
- увеличение `t` отдельно для каждого параметра;
- отсутствие bias correction или квадратного корня над `v_hat`;
- пересоздание состояния на каждом step;
- обновление параметров с включённым autograd.

### Официальные материалы и примеры
- [Adam в `torch.optim`](https://docs.pytorch.org/docs/stable/optim.html#torch.optim.Adam) — формула, параметры и пример использования;
- [Базовый интерфейс optimizer](https://docs.pytorch.org/docs/stable/optim.html#how-to-use-an-optimizer) — жизненный цикл `zero_grad → backward → step`;
- [Zeroing out gradients](https://docs.pytorch.org/tutorials/recipes/recipes/zeroing_out_gradients.html) — почему gradients нужно очищать явно.

### Вопросы для самопроверки
1. Почему `v` хранит квадраты gradients, а не сами gradients?
2. Что именно исправляет bias correction на первых шагах?
3. Чем состояние optimizer отличается от gradient параметра?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        pass  # store params, init m and v to zeros

    def step(self):
        pass  # update params using Adam rule

    def zero_grad(self):
        pass  # zero all gradients

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('adam')